In [2]:
# 필요한 라이브러리 설치 (처음 실행 시)
# moviepy는 이제 필요 없으므로 whisper와 tqdm만 설치하면 됩니다.
# !pip install openai-whisper tqdm

import os
import sys
import whisper
from tqdm import tqdm
import subprocess  # FFmpeg를 직접 호출하기 위해 subprocess 라이브러리를 추가합니다.

# 1. 기본 설정
video_dir = r"C:\Temp\ti_movie"
output_dir = os.path.join(video_dir, "transcripts")

# 디렉토리 존재 여부 확인
if not os.path.exists(video_dir):
    print(f"❌ 비디오 디렉토리가 존재하지 않습니다: {video_dir}")
    print("올바른 경로를 확인해주세요.")
    sys.exit(1)

# 출력 디렉토리 생성
os.makedirs(output_dir, exist_ok=True)

# 2. Whisper 모델 불러오기 (기본 small, 더 정확하게 하려면 'medium' 또는 'large')
try:
    print("🤖 Whisper 모델을 로딩중...")
    model = whisper.load_model("small")
    print("✅ 모델 로딩 완료!")
except Exception as e:
    print(f"❌ Whisper 모델 로딩 실패: {e}")
    sys.exit(1)

# 3. 폴더 내 영상 파일 탐색
try:
    video_files = [f for f in os.listdir(video_dir)
                   if f.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv','.ts'))]

    if not video_files:
        print(f"❌ {video_dir} 폴더에서 영상 파일을 찾을 수 없습니다.")
        print("지원 형식: .mp4, .mkv, .avi, .mov, .wmv, .flv, .ts")
        sys.exit(1)

    print(f"📁 발견된 영상 파일: {len(video_files)}개")
    for i, file in enumerate(video_files, 1):
        print(f"  {i}. {file}")
    print()

except Exception as e:
    print(f"❌ 파일 탐색 중 오류 발생: {e}")
    sys.exit(1)

# 4. 영상 처리
for video_file in tqdm(video_files, desc="🎧 Processing videos"):
    try:
        print(f"\n🎬 처리중: {video_file}")

        video_path = os.path.join(video_dir, video_file)
        base_name = os.path.splitext(video_file)[0]
        audio_path = os.path.join(video_dir, f"{base_name}.wav")
        output_path = os.path.join(output_dir, f"{base_name}.txt")

        # 이미 처리된 파일인지 확인
        if os.path.exists(output_path):
            print(f"⏭️ 이미 처리된 파일입니다: {output_path}")
            continue

        # --- ✨ 5. 영상에서 오디오 추출 (FFmpeg 직접 호출 방식으로 변경) ---
        print("  🔊 오디오 추출중 (FFmpeg 사용)...")
        
        # FFmpeg 명령어 리스트 정의
        # -i: 입력 파일(비디오), -vn: 비디오 스트림 제외, -acodec: 오디오 코덱 지정, -y: 같은 이름의 파일이 있으면 덮어쓰기
        command = [
            'ffmpeg',
            '-i', video_path,
            '-vn',
            '-acodec', 'pcm_s16le', # Whisper가 잘 인식하는 wav 형식 코덱
            '-ar', '16000',        # 샘플 레이트를 16000Hz로 설정 (Whisper 권장)
            '-ac', '1',            # 오디오 채널을 1개(모노)로 설정
            '-y',
            audio_path
        ]
        
        # FFmpeg 명령어 실행. 화면에 로그를 출력하지 않도록 설정.
        # check=True는 명령어 실행 실패 시 오류를 발생시킵니다.
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 6. Whisper로 음성 텍스트 변환
        print("  🗣️ 음성 인식중...")
        result = model.transcribe(audio_path, language="en")

        # 7. 텍스트 저장
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(result["text"])

        print(f"  ✅ 완료: {output_path}")

        # 8. 임시 오디오 파일 삭제
        if os.path.exists(audio_path):
            os.remove(audio_path)

    except subprocess.CalledProcessError:
        # FFmpeg 실행이 실패한 경우 (예: 오디오 트랙이 없는 영상)
        print(f"  ❌ 오류 발생 ({video_file}): 오디오 트랙을 추출할 수 없습니다.")
        continue
    except Exception as e:
        print(f"  ❌ 오류 발생 ({video_file}): {str(e)}")
        # 임시 파일 정리
        if 'audio_path' in locals() and os.path.exists(audio_path):
            try:
                os.remove(audio_path)
            except:
                pass
        continue

print("\n🎉 모든 영상의 음성 텍스트 변환이 완료되었습니다!")
print(f"📂 결과 파일 위치: {output_dir}")

🤖 Whisper 모델을 로딩중...
✅ 모델 로딩 완료!
📁 발견된 영상 파일: 6개
  1. 2025 BCA Update and Titanium Perspective TITANIUM USA 2025 Bosto.mp4
  2. Airbus Global Market Forecast.ts
  3. Global Industrial Markets - Moderator_Nate Fairfield, Titanium Industries.ts
  4. Titanium Industry World Supply Trends Moderator Andy Bayne, TIMET.ts
  5. World Demand Trends-Ryosaku Kadowaki Presentation.ts
  6. World Titanium Industry Demand Trends Moderator Peter Zimm Charl.mp4



🎧 Processing videos:   0%|          | 0/6 [00:00<?, ?it/s]


🎬 처리중: 2025 BCA Update and Titanium Perspective TITANIUM USA 2025 Bosto.mp4
  🔊 오디오 추출중 (FFmpeg 사용)...
  🗣️ 음성 인식중...


🎧 Processing videos:  17%|█▋        | 1/6 [09:01<45:07, 541.48s/it]

  ✅ 완료: C:\Temp\ti_movie\transcripts\2025 BCA Update and Titanium Perspective TITANIUM USA 2025 Bosto.txt

🎬 처리중: Airbus Global Market Forecast.ts
  🔊 오디오 추출중 (FFmpeg 사용)...
  🗣️ 음성 인식중...


...

  ✅ 완료: C:\Temp\ti_movie\transcripts\Titanium Industry World Supply Trends Moderator Andy Bayne, TIMET.txt

🎬 처리중: World Demand Trends-Ryosaku Kadowaki Presentation.ts
  🔊 오디오 추출중 (FFmpeg 사용)...
  🗣️ 음성 인식중...


🎧 Processing videos:  83%|████████▎ | 5/6 [37:16<06:58, 418.20s/it]

  ✅ 완료: C:\Temp\ti_movie\transcripts\World Demand Trends-Ryosaku Kadowaki Presentation.txt

🎬 처리중: World Titanium Industry Demand Trends Moderator Peter Zimm Charl.mp4
  🔊 오디오 추출중 (FFmpeg 사용)...
  🗣️ 음성 인식중...


🎧 Processing videos: 100%|██████████| 6/6 [54:10<00:00, 541.82s/it]

  ✅ 완료: C:\Temp\ti_movie\transcripts\World Titanium Industry Demand Trends Moderator Peter Zimm Charl.txt

🎉 모든 영상의 음성 텍스트 변환이 완료되었습니다!
📂 결과 파일 위치: C:\Temp\ti_movie\transcripts
